In [1]:
#!pip install ragatouille==0.0.9post2 "langchain==0.1.14" faiss-cpu



In [2]:

# Enhanced RAG Pipeline with Query Augmentation
# Based on original Rag_for_dataset.ipynb with query rewriting enhancement
# Optimized for A100 GPU on Lightning AI

import json
from ragatouille import RAGPretrainedModel
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from tqdm import tqdm
import torch
import warnings
warnings.filterwarnings('ignore')

/tmp/ipykernel_9846/1331358883.py:6: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel


In [3]:
# import zipfile
# import os

# def unzip_folder(zip_path, extract_to=None):
#     """
#     Unzips a .zip file to the specified directory.
    
#     Args:
#         zip_path (str): Path to the .zip file.
#         extract_to (str, optional): Destination folder to extract files to.
#                                     If None, extracts in the same location as the zip file.
#     """
#     zip_path = os.path.abspath(zip_path)
#     if extract_to is None:
#         extract_to = os.path.splitext(zip_path)[0]  # create a folder with same name as zip

#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         zip_ref.extractall(extract_to)
#         print(f"✅ Extracted all files to: {extract_to}")


# # Example usage:
# unzip_folder("tqa_colbert_index.zip")


In [23]:
# ============================================
# CONFIGURATION
# ============================================

# File paths
CONTEXT_FILE = "all_context.jsonl"
QUESTION_FILE = "valid_questions.jsonl"  # Change to train/val/test as needed
OUTPUT_FILE = "valid_finetune_with_context_and_query_aug.jsonl"  # Change accordingly

# Fine-tuned query rewriter model path
QUERY_REWRITER_PATH = "../flan_t5_query_augmentation"

# Retrieval settings
TOP_K = 1  # Number of contexts to retrieve PER QUERY (original + rewritten)
INDEX_NAME = "tqa_colbert_index/tqa_colbert_index"  # Name for the ColBERT index
MAX_DOC_LENGTH = 512  # Max passage size is 510 tokens, so 512 is perfect

# A100 GPU Configuration
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_FP16 = True  # A100 supports FP16 efficiently
BATCH_REWRITE = True  # Batch process rewrites for speed
REWRITE_BATCH_SIZE = 32  # Optimal for A100

In [24]:
# Verify GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print("=" * 80)
    print("ENHANCED RAG WITH QUERY AUGMENTATION - A100 OPTIMIZED")
    print("=" * 80)
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_memory:.2f} GB")
    print(f"Device: {DEVICE}")
    print(f"FP16 Enabled: {USE_FP16}")
    print(f"Query rewriter: {QUERY_REWRITER_PATH}")
    print(f"Top-{TOP_K} passages per query (original + rewritten)")
    print(f"Batch rewriting: {BATCH_REWRITE} (batch size: {REWRITE_BATCH_SIZE})")
    print("=" * 80 + "\n")
else:
    print("WARNING: GPU not detected!")
    USE_FP16 = False
    BATCH_REWRITE = False

ENHANCED RAG WITH QUERY AUGMENTATION - A100 OPTIMIZED
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.29 GB
Device: cuda
FP16 Enabled: True
Query rewriter: ../flan_t5_query_augmentation
Top-1 passages per query (original + rewritten)
Batch rewriting: True (batch size: 32)



In [25]:
# ============================================
# STEP 1: Load Context Data
# ============================================

def load_jsonl(file_path):
    """Load JSONL file into a list of dictionaries"""
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line.strip()))
    return data

print("Loading context data...")
contexts = load_jsonl(CONTEXT_FILE)
print(f"✓ Loaded {len(contexts)} passages from {CONTEXT_FILE}\n")


Loading context data...
✓ Loaded 6810 passages from all_context.jsonl



In [26]:
# ============================================
# STEP 2: Load Query Rewriter Model (A100 Optimized)
# ============================================

print("Loading fine-tuned query rewriter model...")
rewriter_tokenizer = AutoTokenizer.from_pretrained(QUERY_REWRITER_PATH)

# Load with FP16 for A100 efficiency
if USE_FP16:
    rewriter_model = AutoModelForSeq2SeqLM.from_pretrained(
        QUERY_REWRITER_PATH,
        torch_dtype=torch.float16
    ).to(DEVICE)
    print(f"✓ Query rewriter loaded on {DEVICE} with FP16")
else:
    rewriter_model = AutoModelForSeq2SeqLM.from_pretrained(QUERY_REWRITER_PATH).to(DEVICE)
    print(f"✓ Query rewriter loaded on {DEVICE}")

rewriter_model.eval()  # Set to evaluation mode
print()

def rewrite_query(question, max_length=256):
    """
    Rewrite a single question using the fine-tuned FLAN-T5 model
    
    Args:
        question: Original question string
        max_length: Maximum length of rewritten question
    
    Returns:
        Rewritten question string
    """
    input_text = f"Rewrite the following question without changing its meaning. question: \n{question}"
    inputs = rewriter_tokenizer(
        input_text, 
        return_tensors="pt", 
        max_length=256,
        truncation=True
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = rewriter_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            temperature=0.7,
        )
    
    rewritten = rewriter_tokenizer.decode(outputs[0], skip_special_tokens=True)
    return rewritten


def rewrite_queries_batch(questions, max_length=256):
    """
    Batch rewrite questions for A100 efficiency
    
    Args:
        questions: List of question strings
        max_length: Maximum length of rewritten questions
    
    Returns:
        List of rewritten question strings
    """
    input_texts = [
        f"Rewrite the following question without changing its meaning. question: \n{q}" 
        for q in questions
    ]
    
    inputs = rewriter_tokenizer(
        input_texts,
        return_tensors="pt",
        max_length=256,
        truncation=True,
        padding=True
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = rewriter_model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            temperature=0.7,
        )
    
    rewritten = rewriter_tokenizer.batch_decode(outputs, skip_special_tokens=True)
    return rewritten


Loading fine-tuned query rewriter model...
✓ Query rewriter loaded on cuda with FP16



In [27]:
# ============================================
# STEP 3: Initialize ColBERT Model (Load existing index)
# ============================================

print(f"Loading ColBERT index '{INDEX_NAME}'...")
# Load the pre-built index instead of building new one
rag = RAGPretrainedModel.from_index(INDEX_NAME)
print(f"✓ ColBERT index '{INDEX_NAME}' loaded\n")


Loading ColBERT index 'tqa_colbert_index/tqa_colbert_index'...
✓ ColBERT index 'tqa_colbert_index/tqa_colbert_index' loaded



In [28]:
# ============================================
# STEP 4: Load Questions
# ============================================

print(f"Loading questions from {QUESTION_FILE}...")
questions = load_jsonl(QUESTION_FILE)
print(f"✓ Loaded {len(questions)} questions\n")


Loading questions from valid_questions.jsonl...
✓ Loaded 2528 questions



In [29]:
# ============================================
# STEP 5: Retrieve Contexts with Query Augmentation (A100 Optimized)
# ============================================

print(f"Retrieving contexts with query augmentation...")
print(f"Strategy: Top-{TOP_K} from original + Top-{TOP_K} from rewritten query")
print(f"Then deduplicate to get max 2 unique passages per question")

if BATCH_REWRITE:
    print(f"Using batch rewriting with batch size {REWRITE_BATCH_SIZE} for A100 efficiency\n")
else:
    print(f"Using sequential rewriting\n")

enhanced_data = []
rewrite_cache = {}  # Optional: save rewrites for analysis

if BATCH_REWRITE:
    # BATCH PROCESSING (A100 Optimized)
    all_questions = [q["question"] for q in questions]
    
    # Rewrite all questions in batches
    print("Rewriting all questions in batches...")
    all_rewritten = []
    for i in tqdm(range(0, len(all_questions), REWRITE_BATCH_SIZE), desc="Batch rewriting"):
        batch = all_questions[i:i + REWRITE_BATCH_SIZE]
        rewritten_batch = rewrite_queries_batch(batch)
        all_rewritten.extend(rewritten_batch)
    
    # Store rewrites in cache
    rewrite_cache = dict(zip(all_questions, all_rewritten))
    
    # Now process retrieval
    print("\nRetrieving contexts...")
    for idx, q_item in enumerate(tqdm(questions, desc="Processing questions")):
        original_question = q_item["question"]
        rewritten_question = all_rewritten[idx]
        
        # Search with ORIGINAL question (top-1)
        results_original = rag.search(original_question, k=TOP_K)
        
        # Search with REWRITTEN question (top-1)
        results_rewritten = rag.search(rewritten_question, k=TOP_K)
        
        # Combine and deduplicate contexts
        unique_contexts = {}
        
        # Add passages from original query
        for result in results_original:
            content = result["content"]
            if content not in unique_contexts:
                unique_contexts[content] = result
        
        # Add passages from rewritten query
        for result in results_rewritten:
            content = result["content"]
            if content not in unique_contexts:
                unique_contexts[content] = result
        
        # Combine retrieved contexts into single string
        retrieved_texts = [ctx["content"] for ctx in unique_contexts.values()]
        combined_context = "\n\n".join(retrieved_texts)
        
        # Create enhanced entry (WITHOUT rewritten_question)
        enhanced_entry = {
            "question": original_question,
            "answerChoices": q_item["answerChoices"],
            "correctAnswer": q_item["correctAnswer"],
            "context": combined_context
        }
        
        enhanced_data.append(enhanced_entry)

else:
    # SEQUENTIAL PROCESSING (fallback)
    for q_item in tqdm(questions, desc="Processing questions"):
        original_question = q_item["question"]
        
        # Rewrite the question
        rewritten_question = rewrite_query(original_question)
        rewrite_cache[original_question] = rewritten_question
        
        # Search with ORIGINAL question (top-1)
        results_original = rag.search(original_question, k=TOP_K)
        
        # Search with REWRITTEN question (top-1)
        results_rewritten = rag.search(rewritten_question, k=TOP_K)
        
        # Combine and deduplicate contexts
        unique_contexts = {}
        
        # Add passages from original query
        for result in results_original:
            content = result["content"]
            if content not in unique_contexts:
                unique_contexts[content] = result
        
        # Add passages from rewritten query
        for result in results_rewritten:
            content = result["content"]
            if content not in unique_contexts:
                unique_contexts[content] = result
        
        # Combine retrieved contexts into single string
        retrieved_texts = [ctx["content"] for ctx in unique_contexts.values()]
        combined_context = "\n\n".join(retrieved_texts)
        
        # Create enhanced entry (WITHOUT rewritten_question)
        enhanced_entry = {
            "question": original_question,
            "answerChoices": q_item["answerChoices"],
            "correctAnswer": q_item["correctAnswer"],
            "context": combined_context
        }
        
        enhanced_data.append(enhanced_entry)

print(f"✓ Retrieved contexts for all {len(questions)} questions\n")


Retrieving contexts with query augmentation...
Strategy: Top-1 from original + Top-1 from rewritten query
Then deduplicate to get max 2 unique passages per question
Using batch rewriting with batch size 32 for A100 efficiency

Rewriting all questions in batches...


Batch rewriting: 100%|██████████| 79/79 [02:16<00:00,  1.72s/it]



Retrieving contexts...


Processing questions:   0%|          | 0/2528 [00:00<?, ?it/s]

Loading searcher for index tqa_colbert_index for the first time... This may take a few seconds
[Nov 09, 18:33:56] #> Loading codec...
[Nov 09, 18:33:56] #> Loading IVF...
[Nov 09, 18:33:56] #> Loading doclens...


...

[Nov 09, 18:33:56] #> Loading codes and residuals...




Processing questions:   0%|          | 1/2528 [00:01<1:00:38,  1.44s/it]

Searcher loaded!

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: Gravity causes erosion by all of the following except, 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  8992,  5320, 14173,  2011,  2035,  1997,  1996,  2206,
         3272,   102,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')



Processing questions: 100%|██████████| 2528/2528 [02:20<00:00, 17.96it/s]

✓ Retrieved contexts for all 2528 questions



In [30]:
# ============================================
# STEP 6: Save Enhanced Dataset
# ============================================

print(f"Saving enhanced dataset to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    for item in enhanced_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"✓ Saved {len(enhanced_data)} question-answer pairs with contexts\n")


Saving enhanced dataset to valid_finetune_with_context_and_query_aug.jsonl...
✓ Saved 2528 question-answer pairs with contexts



In [31]:
# ============================================
# SUMMARY
# ============================================

print("=" * 50)
print("PIPELINE COMPLETE!")
print("=" * 50)

# Calculate statistics
avg_passages = sum(len(item["context"].split("\n\n")) for item in enhanced_data) / len(enhanced_data)

print(f"\nOutput file: {OUTPUT_FILE}")
print(f"Total samples: {len(enhanced_data)}")
print(f"Average passages per question: {avg_passages:.2f}")
print(f"\nEach entry contains:")
print("  - question (original only)")
print("  - answerChoices")
print("  - correctAnswer")
print("  - context (top-1 from original + top-1 from rewritten, deduplicated)")

print("\n" + "=" * 50)
print("SAMPLE OUTPUT:")
print("=" * 50)
sample = enhanced_data[0]
print(json.dumps(sample, indent=2, ensure_ascii=False)[:800] + "...")
print(f"\nRewritten query: {rewrite_cache[sample['question']][:150]}...")
print("=" * 50)

PIPELINE COMPLETE!

Output file: valid_finetune_with_context_and_query_aug.jsonl
Total samples: 2528
Average passages per question: 1.52

Each entry contains:
  - question (original only)
  - answerChoices
  - correctAnswer
  - context (top-1 from original + top-1 from rewritten, deduplicated)

SAMPLE OUTPUT:
{
  "question": "Gravity causes erosion by all of the following except",
  "answerChoices": "(A) glaciers.. (B) moving air.. (C) flowing water.. (D) mass movement..",
  "correctAnswer": "B",
  "context": "Gravity is responsible for erosion by flowing water and glaciers. Thats because gravity pulls water and ice downhill. These are ways gravity causes erosion indirectly. But gravity also causes erosion directly. Gravity can pull soil, mud, and rocks down cliffs and hillsides. This type of erosion and deposition is called mass movement. It may happen suddenly. Or it may occur very slowly, over many years."
}...

Rewritten query: What processes, not influenced by gravitation, result 